# ĐỒ ÁN MÔN HỌC MÁY: XÂY DỰNG MÔ HÌNH DỰ ĐOÁN NGUY CƠ TIỂU ĐƯỜNG
---
- **Đề tài:** Dự đoán bệnh nhân tiểu đường dựa trên các chỉ số y tế cơ bản (BMI, Tuổi, Lượng đường trong máu)
- **Bộ dữ liệu:** Pima Indians Diabetes Dataset (Kaggle / UCI)
- **Thuật toán triển khai:** Logistic Regression, Decision Tree, Random Forest, Support Vector Machine (SVM)
- **Kỹ thuật tối ưu:** Scikit-learn Pipeline, GridSearchCV, 5-Fold Stratified Cross-Validation

## BƯỚC 1 & 2: KHỞI TẠO MÔI TRƯỜNG VÀ NẠP DỮ LIỆU

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

# Nạp bộ dữ liệu
data_path = os.path.join("..", "dataset", "diabetes.csv")
if not os.path.exists(data_path):
    data_path = os.path.join("dataset", "diabetes.csv")
df = pd.read_csv(data_path)
print(f"Kích thước dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột")
df.head()

## BƯỚC 3: KHÁM PHÁ VÀ PHÂN TÍCH DỮ LIỆU (EDA)

In [ ]:
# Thống kê mô tả các thuộc tính
print("--- THỐNG KÊ MÔ TẢ ---")
display(df.describe().round(2))

# Phân tích phân bố nhãn Outcome
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Outcome", palette=["#3182ce", "#e53e3e"], hue="Outcome", legend=False)
plt.title("Phân bố nhãn mục tiêu (0: Không tiểu đường, 1: Tiểu đường)", fontweight="bold")
plt.xticks([0, 1], ["Không tiểu đường (0)", "Tiểu đường (1)"])
plt.show()

# Ma trận tương quan Pearson
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="Blues", square=True)
plt.title("Ma trận tương quan Pearson giữa các thuộc tính", fontweight="bold")
plt.show()

## BƯỚC 4: TIỀN XỬ LÝ DỮ LIỆU & QUY ĐỔI GIÁ TRỊ 0 ẨN THÀNH NAN

In [ ]:
# Phát hiện giá trị 0 không hợp lý ở 5 cột lâm sàng
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for col in zero_cols:
    cnt = (df[col] == 0).sum()
    print(f"Cột {col:15}: {cnt} giá trị 0 ({cnt/len(df)*100:.2f}%)")

# Thay thế số 0 thành np.nan
df_cleaned = df.copy()
df_cleaned[zero_cols] = df_cleaned[zero_cols].replace(0, np.nan)
print("\nĐã quy đổi thành công số 0 thành NaN để SimpleImputer tự động xử lý!")

## BƯỚC 5: SO SÁNH 2 KỊCH BẢN ĐẶC TRƯNG (KỊCH BẢN A vs KỊCH BẢN B)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score

y = df_cleaned["Outcome"]

# Kịch bản A: 3 đặc trưng (Glucose, BMI, Age)
Xa = df_cleaned[["Glucose", "BMI", "Age"]]
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(Xa, y, test_size=0.2, random_state=42, stratify=y)
pipe_a = Pipeline([("imp", SimpleImputer(strategy="median")), ("scl", StandardScaler()), ("clf", RandomForestClassifier(random_state=42))])
pipe_a.fit(Xa_tr, ya_tr)
pred_a, prob_a = pipe_a.predict(Xa_te), pipe_a.predict_proba(Xa_te)[:, 1]

# Kịch bản B: 8 đặc trưng
Xb = df_cleaned.drop(columns=["Outcome"])
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, y, test_size=0.2, random_state=42, stratify=y)
pipe_b = Pipeline([("imp", SimpleImputer(strategy="median")), ("scl", StandardScaler()), ("clf", RandomForestClassifier(random_state=42))])
pipe_b.fit(Xb_tr, yb_tr)
pred_b, prob_b = pipe_b.predict(Xb_te), pipe_b.predict_proba(Xb_te)[:, 1]

comp_df = pd.DataFrame({
    "Chỉ số": ["Accuracy (%)", "Recall (%)", "F1-Score (%)", "ROC-AUC (%)"],
    "Kịch bản A (3 đặc trưng)": [accuracy_score(ya_te, pred_a)*100, recall_score(ya_te, pred_a)*100, f1_score(ya_te, pred_a)*100, roc_auc_score(ya_te, prob_a)*100],
    "Kịch bản B (8 đặc trưng)": [accuracy_score(yb_te, pred_b)*100, recall_score(yb_te, pred_b)*100, f1_score(yb_te, pred_b)*100, roc_auc_score(yb_te, prob_b)*100]
}).round(2)
display(comp_df)

## BƯỚC 6 & 7: CHIA TẬP TRAIN/TEST VÀ HUẤN LUYỆN 4 THUẬT TOÁN

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

features = ["Glucose", "BMI", "Age"]
X = df_cleaned[features]
y = df_cleaned["Outcome"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=4),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=100),
    "Support Vector Machine (SVM)": SVC(random_state=42, kernel="rbf", probability=True)
}

trained_pipes = {}
for name, clf in models.items():
    pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("clf", clf)])
    pipe.fit(X_train, y_train)
    trained_pipes[name] = pipe
    print(f"✅ Đã huấn luyện xong: {name}")

## BƯỚC 8: ĐÁNH GIÁ MÔ HÌNH (ACCURACY, PRECISION, RECALL, F1, ROC-AUC)

In [ ]:
from sklearn.metrics import precision_score, confusion_matrix, roc_curve

res_eval = {}
plt.figure(figsize=(7, 5))
for name, pipe in trained_pipes.items():
    ypred = pipe.predict(X_test)
    yprob = pipe.predict_proba(X_test)[:, 1]
    res_eval[name] = {
        "Accuracy (%)": round(accuracy_score(y_test, ypred)*100, 2),
        "Precision (%)": round(precision_score(y_test, ypred)*100, 2),
        "Recall (%)": round(recall_score(y_test, ypred)*100, 2),
        "F1-Score (%)": round(f1_score(y_test, ypred)*100, 2),
        "ROC-AUC (%)": round(roc_auc_score(y_test, yprob)*100, 2)
    }
    fpr, tpr, _ = roc_curve(y_test, yprob)
    plt.plot(fpr, tpr, label=f"{name} (AUC={res_eval[name]['ROC-AUC (%)']}%)")

plt.plot([0, 1], [0, 1], "k--")
plt.title("Đường cong ROC Curve so sánh 4 mô hình Machine Learning", fontweight="bold")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.legend()
plt.show()

display(pd.DataFrame(res_eval).T)

## BƯỚC 9: TỐI ƯU SIÊU THAM SỐ VỚI GRIDSEARCHCV & STRATIFIED K-FOLD

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

dt_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("clf", DecisionTreeClassifier(random_state=42))])
param_grid = {
    "clf__max_depth": [3, 4, 5, 6],
    "clf__min_samples_split": [5, 10, 15],
    "clf__min_samples_leaf": [2, 4, 6],
    "clf__criterion": ["gini", "entropy"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(dt_pipe, param_grid, scoring="f1", cv=cv, n_jobs=1)
grid.fit(X_train, y_train)

best_dt = grid.best_estimator_
print("Siêu tham số tối ưu tìm được:", grid.best_params_)

pred_best = best_dt.predict(X_test)
prob_best = best_dt.predict_proba(X_test)[:, 1]
print(f"\n🔥 KẾT QUẢ DECISION TREE SAU TỐI ƯU:")
print(f"- Accuracy : {accuracy_score(y_test, pred_best)*100:.2f}%")
print(f"- Recall   : {recall_score(y_test, pred_best)*100:.2f}%")
print(f"- F1-Score : {f1_score(y_test, pred_best)*100:.2f}%")
print(f"- ROC-AUC  : {roc_auc_score(y_test, prob_best)*100:.2f}%")

## BƯỚC 10 & 11: ĐÓNG GÓI MÔ HÌNH VÀ DỰ ĐOÁN BỆNH NHÂN MỚI

In [ ]:
# Lưu mô hình vào models/diabetes_model.pkl
os.makedirs("../models", exist_ok=True)
joblib.dump(best_dt, "../models/diabetes_model.pkl")
print("✅ Đã lưu thành công mô hình tối ưu vào models/diabetes_model.pkl!")

# Thử nghiệm dự đoán bệnh nhân mới
sample_patient = pd.DataFrame([[165.0, 34.0, 52]], columns=features)
prediction = best_dt.predict(sample_patient)[0]
probability = best_dt.predict_proba(sample_patient)[0]

print("\n--- KẾT QUẢ DỰ ĐOÁN CA BỆNH THỬ NGHIỆM ---")
print("Nhãn dự đoán:", "CÓ NGUY CƠ TIỂU ĐƯỜNG (1)" if prediction == 1 else "NGUY CƠ THẤP (0)")
print(f"Độ tin cậy: {probability[prediction]*100:.2f}%")